# Quadtree height surface: maps, STL, roughness CSV

Generates a synthetic height map, exports **height**, **normal**, and **roughness** images and a **quadtree** mesh as **STL** (`max_subdivisions=12`) into `data/`, then writes `data/roughness_summary.csv` (metrology from heights + statistics from the saved roughness image).

From a clone, you can replace the first cell with `pip install -e .` plus `pandas` instead of PyPI `truemapdata`.

In [ ]:
%pip install -q truemapdata pandas

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from tmd.image import export_height_map, export_normal_map, export_roughness_map
from tmd.model.base import export_heightmap_to_model
from tmd.surface.metadata import analyze_surface_roughness
from tmd.surface.terrain import TMDTerrain

def truemap_repo_root(start: Path | None = None) -> Path:
    """Resolve TrueMapData repo root from cwd (works from repo root or notebooks/)."""
    p = (start or Path.cwd()).resolve()
    for d in (p, *p.parents):
        if (d / "examples").is_dir() or (d / "tmd").is_dir():
            return d
        if (d / ".git").is_dir():
            return d
    if p.name == "notebooks":
        return p.parent
    return p

REPO_ROOT = truemap_repo_root()

DATA_DIR = REPO_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

HEIGHT_PATH = DATA_DIR / "heightmap.png"
NORMAL_PATH = DATA_DIR / "normal.png"
ROUGHNESS_PATH = DATA_DIR / "roughness.png"
STL_PATH = DATA_DIR / "surface_quadtree.stl"
CSV_PATH = DATA_DIR / "roughness_summary.csv"


In [ ]:
# Synthetic terrain (Perlin uses Python loops; 96x96 keeps this cell responsive.)
W = H = 96
SEED = 42

height_map = TMDTerrain.create_sample_height_map(
    width=W,
    height=H,
    pattern="perlin",
    seed=SEED,
    wave_height=1.0,
    z_value=0.0,
)
height_map = np.asarray(height_map, dtype=np.float64)

export_height_map(height_map, str(HEIGHT_PATH))
export_normal_map(height_map, str(NORMAL_PATH), strength=1.0)
export_roughness_map(height_map, str(ROUGHNESS_PATH), kernel_size=3, scale=1.0)

print("Saved:", HEIGHT_PATH, NORMAL_PATH, ROUGHNESS_PATH, sep="\n  ")

In [ ]:
out = export_heightmap_to_model(
    height_map,
    str(STL_PATH),
    "stl",
    triangulation_method="quadtree",
    max_subdivisions=12,
    max_triangles=120_000,
    error_threshold=0.02,
    z_scale=1.0,
    x_length=1.0,
    y_length=1.0,
    binary=True,
    save_heightmap=False,
)
print("STL:", out or STL_PATH)

In [ ]:
rough_metrology = analyze_surface_roughness(height_map)

img = Image.open(ROUGHNESS_PATH).convert("L")
arr = np.asarray(img, dtype=np.float64) / 255.0
rough_image_stats = {
    "roughness_png_mean": float(arr.mean()),
    "roughness_png_std": float(arr.std()),
    "roughness_png_min": float(arr.min()),
    "roughness_png_max": float(arr.max()),
}

row = {
    "height_w": W,
    "height_h": H,
    "seed": SEED,
    **{f"height_{k}": v for k, v in rough_metrology.items()},
    **rough_image_stats,
}
pd.DataFrame([row]).to_csv(CSV_PATH, index=False)
print("Wrote", CSV_PATH)
pd.read_csv(CSV_PATH).T